In [30]:
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import InMemorySaver
from dotenv import load_dotenv

In [31]:
load_dotenv()

True

In [32]:
class State(TypedDict):
    email: str
    message: str
    approved: bool

In [33]:
llm = ChatOpenAI(model="gpt-4o-mini")

In [34]:
def check_email(state: State):
    print("Email ID:", state["email"])

    if "@" not in state["email"]:
        print("Invalid email ID.")
        return {"approved": False}

    answer = interrupt(
        f"Approve this email ID: {state['email']}? (yes/no)"
    )

    if answer.lower() == "yes":
        return {"approved": True}

    return {"approved": False}

In [35]:
def draft_email(state: State):
    response = llm.invoke(
        f"Write a short professional email to {state['email']} "
        f"about confirming an appointment."
    )

    print("\nDraft Email:")
    print(response.content)

    return {
        "message": response.content
    }

In [36]:
def forward_email(state: State):
    print("\nForwarding email...")
    print("To:", state["email"])
    print("Message:", state["message"])
    print("\nEmail forwarded successfully!")

    return {}

In [37]:
def route_email(state: State):
    if state["approved"]:
        return "draft_email"

    return END

In [38]:
builder = StateGraph(State)

In [39]:
builder.add_node("check_email", check_email)
builder.add_node("draft_email", draft_email)
builder.add_node("forward_email", forward_email)

In [40]:
builder.add_edge(START, "check_email")

In [41]:
builder.add_conditional_edges(
    "check_email",
    route_email,
    {
        "draft_email": "draft_email",
        END: END
    }
)

In [42]:
builder.add_edge("draft_email", "forward_email")
builder.add_edge("forward_email", END)

In [43]:
graph = builder.compile(
    checkpointer=InMemorySaver()
)

In [60]:
config1 = {
    "configurable": {
        "thread_id": "email-1"
    }
}

In [61]:
graph.invoke(
    {
        "email": "user@example.com",
        "message": "",
        "approved": False
    },
    config1
)

Email ID: user@example.com


{'email': 'user@example.com',
 'message': '',
 'approved': False,
 '__interrupt__': [Interrupt(value='Approve this email ID: user@example.com? (yes/no)', id='f3577183698b1683fd48095e3ad99b3b')]}

In [74]:
answer = input("Approve? yes/no: ")

In [75]:
result= graph.invoke(
    Command(resume=answer),
    config1
)

In [76]:
print("Approved:", result["approved"])

Approved: False


In [77]:
print(graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	check_email(check_email)
	draft_email(draft_email)
	forward_email(forward_email)
	__end__([<p>__end__</p>]):::last
	__start__ --> check_email;
	check_email -.-> __end__;
	check_email -.-> draft_email;
	draft_email --> forward_email;
	forward_email --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [78]:
png_bytes = graph.get_graph().draw_mermaid_png()

with open("email_graph.png", "wb") as f:
    f.write(png_bytes)

print("Graph saved to email_graph.png")

Graph saved to email_graph.png
